<a href="https://colab.research.google.com/github/RaihanArkan4/050825/blob/main/Pengunjung_Mall_kelompok.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
import io

# Pengaturan tampilan grafik
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("Library berhasil diimport!")

Library berhasil diimport!


In [ ]:
# Upload dataset CSV
uploaded = files.upload()

# Membaca dataset
nama_file = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[nama_file]))

# Menampilkan data
print("Dataset berhasil dimuat!")
display(df.head())

In [ ]:
# Informasi dataset
print("=== INFORMASI DATASET ===")
print("Jumlah Baris:", df.shape[0])
print("Jumlah Kolom:", df.shape[1])

print("\n=== NAMA KOLOM ===")
print(df.columns.tolist())

print("\n=== TIPE DATA ===")
print(df.dtypes)

print("\n=== INFORMASI LENGKAP ===")
df.info()

In [ ]:
# Mengecek data kosong
print("=== DATA KOSONG ===")
print(df.isnull().sum())

# Mengecek data duplikat
print("\nJumlah Data Duplikat:", df.duplicated().sum())

# Menghapus data duplikat
df = df.drop_duplicates()

# Mengisi data kosong pada kolom numerik dengan median
kolom_numerik = df.select_dtypes(include=np.number).columns

for kolom in kolom_numerik:
    df[kolom] = df[kolom].fillna(df[kolom].median())

# Mengisi data kosong pada kolom kategori dengan modus
kolom_kategori = df.select_dtypes(include='object').columns

for kolom in kolom_kategori:
    if df[kolom].isnull().any():
        df[kolom] = df[kolom].fillna(df[kolom].mode()[0])

print("\n=== HASIL PEMBERSIHAN ===")
print("Jumlah data setelah dibersihkan:", len(df))
print("Sisa data kosong:", df.isnull().sum().sum())

In [ ]:
print("=== STATISTIK DESKRIPTIF ===")
display(df.describe(include='all'))

# Statistik kolom numerik
print("\n=== RATA-RATA DATA NUMERIK ===")
display(df.select_dtypes(include=np.number).mean())

print("\n=== MEDIAN DATA NUMERIK ===")
display(df.select_dtypes(include=np.number).median())

In [ ]:
# Mencari kolom berdasarkan nama
def cari_kolom(kata_kunci):
    for kolom in df.columns:
        if any(k in kolom.lower().replace("_", " ") for k in kata_kunci):
            return kolom
    return None

kolom_numerik = df.select_dtypes(include=np.number).columns.tolist()

kolom_pengunjung = cari_kolom([
    'pengunjung', 'visitor', 'visitors', 'jumlah orang',
    'jumlah pengunjung', 'footfall'
])

kolom_waktu = cari_kolom([
    'tanggal', 'date', 'bulan', 'month', 'waktu', 'time'
])

kolom_kategori = cari_kolom([
    'kategori', 'category', 'jenis', 'gender',
    'kelompok', 'segmen', 'hari', 'day'
])

# Jika kolom pengunjung tidak ditemukan,
# gunakan kolom numerik pertama
if kolom_pengunjung is None and len(kolom_numerik) > 0:
    kolom_pengunjung = kolom_numerik[0]

print("Kolom pengunjung:", kolom_pengunjung)
print("Kolom waktu:", kolom_waktu)
print("Kolom kategori:", kolom_kategori)

In [ ]:
print("=== LAPORAN STATISTIK PENGUNJUNG MALL ===")

if kolom_pengunjung:
    print("Total Pengunjung:",
          f"{df[kolom_pengunjung].sum():,.0f}")

    print("Rata-rata Pengunjung:",
          f"{df[kolom_pengunjung].mean():,.2f}")

    print("Pengunjung Terbanyak:",
          f"{df[kolom_pengunjung].max():,.0f}")

    print("Pengunjung Terendah:",
          f"{df[kolom_pengunjung].min():,.0f}")

    print("Median Pengunjung:",
          f"{df[kolom_pengunjung].median():,.2f}")

if kolom_kategori:
    print("\n=== KATEGORI TERBANYAK ===")
    print(df[kolom_kategori].value_counts().head(5))

if kolom_waktu:
    print("\n=== DATA WAKTU ===")
    print(df[kolom_waktu].value_counts().head(5))

In [ ]:
if kolom_kategori:
    data_bar = df.groupby(kolom_kategori)[kolom_pengunjung].sum().sort_values(ascending=False)
else:
    data_bar = df[kolom_pengunjung].value_counts().head(10)

plt.figure(figsize=(10, 6))
data_bar.plot(kind='bar', color='skyblue')

plt.title('Jumlah Pengunjung Mall Berdasarkan Kategori')
plt.xlabel('Kategori')
plt.ylabel('Jumlah Pengunjung')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Insight:")
print("Grafik menunjukkan kategori dengan jumlah pengunjung tertinggi.")
print("Mall dapat memprioritaskan promosi pada kategori yang memiliki jumlah pengunjung tinggi.")

In [ ]:
if kolom_waktu:
    df_line = df.copy()

    try:
        df_line['_tanggal'] = pd.to_datetime(df_line[kolom_waktu])
        data_line = df_line.groupby(
            df_line['_tanggal'].dt.date
        )[kolom_pengunjung].sum().sort_index()
    except:
        data_line = df_line.groupby(kolom_waktu)[kolom_pengunjung].sum()
else:
    data_line = df[kolom_pengunjung].reset_index(drop=True)

plt.figure(figsize=(12, 6))
plt.plot(data_line.index.astype(str), data_line.values,
         marker='o', color='green')

plt.title('Tren Jumlah Pengunjung Mall')
plt.xlabel('Periode')
plt.ylabel('Jumlah Pengunjung')
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

print("Insight:")
print("Grafik memperlihatkan perubahan jumlah pengunjung.")
print("Periode dengan jumlah pengunjung tinggi dapat dimanfaatkan untuk mengadakan promosi.")

In [ ]:
if len(kolom_numerik) >= 2:
    x_col = kolom_numerik[0]
    y_col = kolom_numerik[1]

    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df, x=x_col, y=y_col, alpha=0.7)

    plt.title('Hubungan Antar Variabel Pengunjung Mall')
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.tight_layout()
    plt.show()

    print("Insight:")
    print(f"Grafik menunjukkan hubungan antara {x_col} dan {y_col}.")
    print("Pola titik dapat digunakan untuk melihat hubungan atau kecenderungan antarvariabel.")
else:
    print("Tidak cukup kolom numerik untuk membuat Scatter Plot.")

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(df[kolom_pengunjung], bins=20, kde=True, color='orange')

plt.title('Distribusi Jumlah Pengunjung Mall')
plt.xlabel('Jumlah Pengunjung')
plt.ylabel('Frekuensi')
plt.tight_layout()
plt.show()

print("Insight:")
print("Histogram menunjukkan persebaran jumlah pengunjung.")
print("Sebaran data dapat membantu mengetahui jumlah pengunjung yang paling sering terjadi.")

In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(y=df[kolom_pengunjung], color='lightblue')

plt.title('Boxplot Jumlah Pengunjung Mall')
plt.xlabel('Data')
plt.ylabel('Jumlah Pengunjung')
plt.tight_layout()
plt.show()

print("Insight:")
print("Boxplot memperlihatkan median, rentang data, dan potensi nilai ekstrem.")
print("Nilai yang berada jauh dari sebaran utama perlu diperiksa lebih lanjut.")

In [ ]:
Q1 = df[kolom_pengunjung].quantile(0.25)
Q3 = df[kolom_pengunjung].quantile(0.75)

IQR = Q3 - Q1

batas_bawah = Q1 - 1.5 * IQR
batas_atas = Q3 + 1.5 * IQR

outlier = df[
    (df[kolom_pengunjung] < batas_bawah) |
    (df[kolom_pengunjung] > batas_atas)
]

print("=== DETEKSI OUTLIER ===")
print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Batas bawah:", batas_bawah)
print("Batas atas:", batas_atas)

print("\nJumlah Outlier:", len(outlier))
display(outlier)

if len(outlier) > 0:
    print("Terdapat data yang berpotensi menjadi outlier.")
    print("Kemungkinan penyebab: acara khusus, hari libur, atau peningkatan aktivitas mall.")
else:
    print("Tidak ditemukan outlier berdasarkan aturan IQR.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# Grafik 1: Bar Chart
if kolom_kategori:
    data = df.groupby(kolom_kategori)[kolom_pengunjung].sum().sort_values(ascending=False).head(10)
    data.plot(kind='bar', ax=axes[0, 0], color='skyblue')
else:
    df[kolom_pengunjung].head(10).plot(kind='bar', ax=axes[0, 0])

axes[0, 0].set_title('Jumlah Pengunjung per Kategori')
axes[0, 0].set_xlabel('Kategori')
axes[0, 0].set_ylabel('Pengunjung')

# Grafik 2: Line Chart
axes[0, 1].plot(range(len(df)),
                df[kolom_pengunjung],
                color='green')
axes[0, 1].set_title('Tren Pengunjung')
axes[0, 1].set_xlabel('Urutan Data')
axes[0, 1].set_ylabel('Pengunjung')

# Grafik 3: Histogram
sns.histplot(df[kolom_pengunjung],
             bins=20,
             ax=axes[1, 0],
             color='orange')
axes[1, 0].set_title('Distribusi Pengunjung')
axes[1, 0].set_xlabel('Pengunjung')
axes[1, 0].set_ylabel('Frekuensi')

# Grafik 4: Boxplot
sns.boxplot(y=df[kolom_pengunjung],
            ax=axes[1, 1],
            color='lightblue')
axes[1, 1].set_title('Sebaran dan Outlier Pengunjung')
axes[1, 1].set_ylabel('Pengunjung')

plt.suptitle('DASHBOARD ANALISIS PENGUNJUNG MALL', fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
total = df[kolom_pengunjung].sum()
rata = df[kolom_pengunjung].mean()
maks = df[kolom_pengunjung].max()
minimum = df[kolom_pengunjung].min()

rekomendasi = f"""
=== LAPORAN ANALISIS PENGUNJUNG MALL ===

Berdasarkan hasil analisis dataset pengunjung mall,
diperoleh total pengunjung sebanyak {total:,.0f} orang.
Rata-rata jumlah pengunjung adalah {rata:,.2f} orang
dengan jumlah tertinggi {maks:,.0f} orang dan
jumlah terendah {minimum:,.0f} orang.

Hasil analisis menunjukkan bahwa jumlah pengunjung
mengalami variasi pada setiap data yang diamati.
Informasi ini dapat digunakan oleh pihak manajemen
untuk mengetahui pola kunjungan dan menentukan
strategi promosi yang lebih tepat.

Manajemen disarankan untuk meningkatkan kegiatan
promosi pada periode yang memiliki jumlah pengunjung
rendah. Program diskon, acara hiburan, dan kerja sama
dengan tenant dapat menjadi pilihan untuk menarik
lebih banyak pengunjung.

Selain itu, periode dengan jumlah pengunjung tinggi
perlu dimanfaatkan untuk mengadakan kegiatan promosi
yang lebih besar. Pihak mall juga perlu memastikan
fasilitas umum, kebersihan, keamanan, dan kenyamanan
pengunjung tetap terjaga.

Data yang memiliki nilai ekstrem perlu diperiksa
lebih lanjut untuk mengetahui apakah disebabkan oleh
acara khusus, hari libur, atau faktor lainnya.

Dengan memanfaatkan hasil analisis ini, Nusantara Mart
diharapkan dapat menyusun strategi promosi yang lebih
efektif dan meningkatkan jumlah kunjungan pada periode
berikutnya.
"""

print(rekomendasi)

In [ ]:
# Menyimpan data yang sudah dibersihkan
df.to_csv('hasil_pembersihan_pengunjung_mall.csv', index=False)

# Download hasil
files.download('hasil_pembersihan_pengunjung_mall.csv')